# Week 5 · Notebook 1  Tokenization Lab

**BPE tokenization, encode/decode, and a token-cost estimator for an OpenAI-compatible API.**

```
# Requirements: pip install tiktoken pandas numpy
```

No internet or API key required  `tiktoken` bundles its tokenizers locally. Part of AI Engineering Lab · ZoroLogistics case study.

## Why this matters

Models do not read characters or words  they read **tokens**. Cost, latency, and the context window are all counted in tokens, so the first practical skill is measuring them with the *actual* tokenizer instead of a `chars / 4` guess. In this notebook you will tokenize real freight text, round-trip encode/decode, compare token counts across text types, and turn a token count into a dollar figure.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root (assumes cwd == week-NN/)
_root = pathlib.Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "zoro").is_dir():
        sys.path.insert(0, str(_p)); break

from zoro import data
import numpy as np, pandas as pd

# Seeded generator of realistic shipment notes (carrier updates, exception logs).
def shipment_notes(n=50, seed=7):
    rng = np.random.default_rng(seed)
    events = [
        "picked up at origin dock",
        "departed {city} sorting hub",
        "arrived {city} intermediate hub",
        "held for customs inspection at {port}",
        "reefer unit {code} fault, temperature excursion logged",
        "driver reassigned after hours-of-service reset",
        "weather hold: {severity} storm on route",
        "delivered to consignee, signed by {name}",
    ]
    cities = ["Chicago", "Memphis", "Houston", "New Orleans", "Seattle", "Portland"]
    ports = ["Long Beach", "Oakland", "Savannah", "Newark"]
    notes = []
    for _ in range(n):
        chosen = rng.choice(events, size=int(rng.integers(2, 6)), replace=False)
        lines = []
        for e in chosen:
            e = e.format(city=rng.choice(cities), port=rng.choice(ports),
                         code=rng.choice(["R1", "R2", "R9", "C4"]),
                         severity=rng.choice(["light", "moderate", "severe"]),
                         name=rng.choice(["J. Doe", "A. Patel", "M. Chen"]))
            lines.append("- " + e)
        notes.append(" ".join(lines))
    return notes

notes = shipment_notes(50, seed=7)
print("generated", len(notes), "shipment notes; first note:")
print(notes[0])

## BPE encode/decode round-trip

BPE builds a vocabulary by repeatedly merging the most frequent adjacent pair of characters. Common words become single tokens; rare words ("ZoroLogistics") split into subwords. `tiktoken` exposes the tokenizers behind OpenAI models; `cl100k_base` is the one used by GPT-4 and `text-embedding-3`.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

text = "ZoroLogistics ships pharmaceutical freight from Houston to New Orleans."
tokens = enc.encode(text)
decoded = enc.decode(tokens)

print("text:        ", text)
print("token ids:   ", tokens)
print("num tokens:  ", len(tokens))
print("roundtrip ok: ", enc.decode(enc.encode(text)) == text)

## Subword splitting on rare words

The same model reads a common word as one token but a brand name or code as several pieces  which is exactly why `chars / 4` estimates drift on freight text full of IDs and carrier codes.

In [ ]:
for s in ["shipment", "ZoroLogistics", "pharmaceuticals", "🚚", "ZRL-10042", "unfathomable"]:
    t = enc.encode(s)
    pieces = [enc.decode([x]) for x in t]
    print(f"{s!r:>22} -> {len(t):2d} token(s): {t}  {pieces}")

## Token counts across text types

The same information tokenizes very differently depending on what it is: prose is compact, IDs and code are token-hungry, and non-ASCII text (emoji, diacritics) inflates the count. Measure, don't guess.

In [ ]:
bol = data.bol_samples(3, seed=5)[0]
tickets = data.support_tickets(50, seed=99, n_shipments=5000)
ticket = tickets.iloc[0]["text"]
code_sample = "def estimate_cost(tokens, price_per_mtok):\n return (tokens / 1_000_000) * price_per_mtok\n"

samples = {
    "English prose": "Freight demand softened in the third quarter while fuel surcharges rose across most lanes.",
    "Shipment note": notes[0],
    "Bill of lading": bol["text"],
    "Support ticket": ticket,
    "Python code": code_sample,
    "Commodity names": ", ".join(data.COMMODITIES),
    "Unicode/emoji": "Shipment 🚚 to München, café ☕ delivered on time.",
}

rows = []
for name, s in samples.items():
    n = len(enc.encode(s))
    rows.append({"text_type": name, "chars": len(s), "tokens": n,
                 "chars_per_token": round(len(s) / n, 1)})
df = pd.DataFrame(rows)
print(df.to_string(index=False))

## The token-cost estimator

Cost is `(input_tokens × $/input_token) + (output_tokens × $/output_token)`, priced per *million* tokens (`$/Mtok`) with output usually several times input. The price is a configurable parameter  swap in your provider's live price.

In [ ]:
def estimate_cost(input_tokens, output_tokens, price_in=1.00, price_out=3.00):
    # prices are USD per 1,000,000 tokens
    return (input_tokens / 1_000_000) * price_in + (output_tokens / 1_000_000) * price_out

system_prompt = "Summarize this shipment note into 3 bullet points for an ops agent."
system_tokens = len(enc.encode(system_prompt))
note_tokens = len(enc.encode(notes[0]))
summary_tokens = 60  # a typical 3-bullet summary

inp = system_tokens + note_tokens
cost = estimate_cost(inp, summary_tokens, price_in=1.00, price_out=3.00)
print(f"system tokens: {system_tokens}, note tokens: {note_tokens}")
print(f"input: {inp} tokens, output: {summary_tokens} tokens")
print(f"per-call cost at $1.00/$3.00 per Mtok: ${cost:.6f}")

## Scale it: 100,000 notes a day

A single number turns "cool demo" into "does this fit the ops budget". The levers it exposes (shorter prompt, capped output, summarize only flagged shipments) are exactly what Week 6 spends on.

In [ ]:
avg_note = sum(len(enc.encode(n)) for n in notes) / len(notes)
per_call_in = system_tokens + avg_note
n_calls = 100_000
daily_cost = n_calls * estimate_cost(per_call_in, summary_tokens, 1.00, 3.00)
print(f"avg note tokens: {avg_note:.0f}, per-call input tokens: {per_call_in:.0f}")
print(f"daily cost over {n_calls:,} notes: ${daily_cost:,.2f}")

In [ ]:
# Week 5 · Notebook 1 headline metric: estimated daily token cost of summarizing
# 100,000 shipment notes at $1.00/Mtok input and $3.00/Mtok output.
print("WEEK5_NB1_DAILY_COST_USD:", round(daily_cost, 2))